In [1]:
"""
=========================================================
06_feature_ablation.py
=========================================================

Purpose
-------
Evaluate the importance of every feature by removing
one feature at a time and measuring performance drop.

Inputs
------
data/processed/feature_engineered.csv
outputs/selected_features.json
outputs/best_hyperparameters.json

Outputs
-------
outputs/feature_ablation_results.csv
"""

'\n=========================================================\n06_feature_ablation.py\n=========================================================\n\nPurpose\n-------\nEvaluate the importance of every feature by removing\none feature at a time and measuring performance drop.\n\nInputs\n------\ndata/processed/feature_engineered.csv\noutputs/selected_features.json\noutputs/best_hyperparameters.json\n\nOutputs\n-------\noutputs/feature_ablation_results.csv\n'

In [2]:
import json
import warnings

import pandas as pd

from catboost import CatBoostClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)
from sklearn.model_selection import train_test_split
from pathlib import Path
warnings.filterwarnings("ignore")

In [3]:
#Global declare random state
RANDOM_STATE = 42

In [4]:
print("=" * 70)
print("FEATURE ABLATION")
print("=" * 70)

FEATURE ABLATION


In [5]:
# ==========================================================
# Paths
# ==========================================================

BASE_DIR = Path.cwd().parent

In [6]:
##########################################################
# LOAD DATA
##########################################################

df = pd.read_csv(f"{BASE_DIR}/data/feature_engineered.csv")

TARGET = "target"

with open(f"{BASE_DIR}/outputs/selected_features.json") as f:
    FEATURES = json.load(f)

with open(f"{BASE_DIR}/outputs/best_hyperparameters.json") as f:
    BEST_PARAMS = json.load(f)

X = df[FEATURES].copy()
y = df[TARGET]

In [11]:
##########################################################
# CATEGORICAL FEATURES
##########################################################

categorical_columns = [
    "Category_Bucket_final",
    "Vertical",
    "self_dealer_status",
    "Dealing_Zone",
    "plan"
]

for col in categorical_columns:
    if col in X.columns:
        X[col] = X[col].fillna("Missing").astype(str)

In [12]:
##########################################################
# TRAIN / VALID / TEST
##########################################################

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

In [13]:
##########################################################
# MODEL PARAMETERS
##########################################################

BEST_PARAMS.update({

    "loss_function": "Logloss",

    "eval_metric": "AUC",

    "random_seed": RANDOM_STATE,

    "verbose": False

})

In [14]:
##########################################################
# BASELINE MODEL
##########################################################

cat_features = [
    X.columns.get_loc(col)
    for col in categorical_columns
    if col in X.columns
]

baseline_model = CatBoostClassifier(**BEST_PARAMS)

baseline_model.fit(

    X_train,
    y_train,

    cat_features=cat_features,

    eval_set=(X_valid, y_valid),

    use_best_model=True,

    verbose=False

)

baseline_prob = baseline_model.predict_proba(X_test)[:,1]

baseline_auc = roc_auc_score(
    y_test,
    baseline_prob
)

baseline_pr = average_precision_score(
    y_test,
    baseline_prob
)

print(f"\nBaseline ROC AUC : {baseline_auc:.6f}")
print(f"Baseline PR AUC  : {baseline_pr:.6f}")


Baseline ROC AUC : 0.595998
Baseline PR AUC  : 0.010275


In [ ]:
##########################################################
# FEATURE ABLATION
##########################################################

results = []

for feature in FEATURES:

    print(f"Removing {feature}")

    remaining = [f for f in FEATURES if f != feature]

    X_train_new = X_train[remaining]
    X_valid_new = X_valid[remaining]
    X_test_new = X_test[remaining]

    cat_new = [
        X_train_new.columns.get_loc(col)
        for col in categorical_columns
        if col in X_train_new.columns
    ]

    model = CatBoostClassifier(**BEST_PARAMS)

    model.fit(

        X_train_new,
        y_train,

        cat_features=cat_new,

        eval_set=(X_valid_new, y_valid),

        use_best_model=True,

        verbose=False

    )

    pred = model.predict_proba(X_test_new)[:,1]

    auc = roc_auc_score(
        y_test,
        pred
    )

    pr = average_precision_score(
        y_test,
        pred
    )

    results.append({

        "Removed Feature": feature,

        "ROC AUC": auc,

        "PR AUC": pr,

        "Delta AUC": baseline_auc - auc

    })

Removing current_age


In [ ]:
##########################################################
# BASELINE ROW
##########################################################

results.append({

    "Removed Feature": "None",

    "ROC AUC": baseline_auc,

    "PR AUC": baseline_pr,

    "Delta AUC": 0

})

##########################################################
# SAVE RESULTS
##########################################################

results = pd.DataFrame(results)

results = results.sort_values(
    "Delta AUC"
)

results.to_csv(

    f"{BASE_DIR}/outputs/feature_ablation_results.csv",

    index=False

)

AttributeError: 'DataFrame' object has no attribute 'append'

In [ ]:
##########################################################
# DISPLAY
##########################################################

print("\n")

print("="*70)
print("FEATURE ABLATION RESULTS")
print("="*70)

print(results)

print("\nSaved")

print(f"{BASE_DIR}/outputs/feature_ablation_results.csv")